# Multi-Agent Travel Planner — CrewAI

The same Singapore classroom example, travel data, tool permissions, specialist instructions and business prompts as the LangGraph and Microsoft notebooks.

Manager requirements and worker selection → selected specialists concurrently → synthesis → critic → at most one revision → final plan.

**CrewAI concepts:** Agent, Task, Crew, LLM, tool, and native Flow state with `@start` / `@listen`.

## Run in Colab
Use a CPU runtime with Python 3.10–3.13. Add `OPENAI_API_KEY` in Colab Secrets and enable Notebook access. Run cells in order. Restart after installation if requested. The demo makes real OpenAI calls and needs API quota. Local Jupyter also supports top-level await. This notebook never books travel.


## Google Colab setup

Use a fresh **Python 3.12 or 3.13 CPU runtime** for each notebook. In the Secrets sidebar, add `OPENAI_API_KEY` and enable **Notebook access** for this notebook. Run cells from top to bottom. If installation requests a restart, restart the session and continue with the imports/credentials cell (do not reinstall repeatedly). No Drive mount or local project files are needed.

The default model is `gpt-4o-mini`. API billing/quota and model access are required; a ChatGPT subscription does not supply API quota. Keys are never printed. Package conflicts can occur if multiple framework notebooks share one runtime—start a fresh session when switching frameworks.


In [ ]:
import sys
if not (3, 10) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("CrewAI requires Python 3.10–3.13. Choose a compatible Colab runtime.")
%pip -q install "crewai==1.15.27" "python-dotenv==1.2.4"


## 1. Runtime and credentials
Colab Secrets first, then an environment variable, then a hidden prompt. Key values never print. The model matches the source.

In [ ]:
import os, asyncio
# Configure before importing CrewAI. Keep classroom tracing disabled.
os.environ["CREWAI_TELEMETRY_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from crewai.flow.flow import Flow, start, listen
from importlib.metadata import version

MODEL = "gpt-4o-mini"

def read_openai_key():
    # In Colab, always use this notebook's enabled secret.
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None
    if userdata is not None:
        try:
            key = userdata.get("OPENAI_API_KEY")
        except Exception:
            raise RuntimeError(
                "In Colab Secrets, add OPENAI_API_KEY and enable Notebook access, "
                "then rerun this cell."
            ) from None
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError("The OPENAI_API_KEY Colab secret is empty.")
        return key.strip()

    # Local Python fallback; never overwrite an existing environment key.
    try:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv(usecwd=True), override=False)
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not key:
        from getpass import getpass
        key = getpass("Enter OPENAI_API_KEY (hidden): ").strip()
    if not key:
        raise ValueError("OPENAI_API_KEY cannot be empty.")
    return key

os.environ["OPENAI_API_KEY"] = read_openai_key()
print("Model:", MODEL)
print("CrewAI:", version("crewai"))


## 2. The same travel data
Copied verbatim from the companion notebook. These are static teaching fixtures. Visa tips, weather and rates are not current advice. Flight prices lack dates/return fares, hotel prices lack room occupancy, and activity admission prices are absent. The plan must state assumptions and unpriced items.

In [ ]:
# ---------- FLIGHTS ----------
FLIGHTS = [
    {"id": "SQ-421", "from": "Mumbai", "to": "Singapore", "airline": "Singapore Airlines",
     "depart": "11:45", "arrive": "19:50", "stops": 0, "price_inr": 28500, "red_eye": False},
    {"id": "AI-342", "from": "Mumbai", "to": "Singapore", "airline": "Air India",
     "depart": "23:15", "arrive": "07:20", "stops": 0, "price_inr": 23600, "red_eye": True},
    {"id": "6E-1071", "from": "Mumbai", "to": "Singapore", "airline": "IndiGo",
     "depart": "08:30", "arrive": "16:40", "stops": 0, "price_inr": 21900, "red_eye": False},
    {"id": "MH-199", "from": "Mumbai", "to": "Singapore", "airline": "Malaysia Airlines",
     "depart": "14:20", "arrive": "23:55", "stops": 1, "price_inr": 19400, "red_eye": False},
    {"id": "SQ-423", "from": "Delhi", "to": "Singapore", "airline": "Singapore Airlines",
     "depart": "10:10", "arrive": "19:05", "stops": 0, "price_inr": 31200, "red_eye": False},
    {"id": "6E-1085", "from": "Bangalore", "to": "Singapore", "airline": "IndiGo",
     "depart": "09:05", "arrive": "16:20", "stops": 0, "price_inr": 20800, "red_eye": False},
]

# ---------- HOTELS ----------
HOTELS = [
    {"name": "Hotel Boss", "city": "Singapore", "area": "Lavender",
     "price_per_night_inr": 7200, "rating": 4.0, "style": "budget"},
    {"name": "Village Hotel Bugis", "city": "Singapore", "area": "Bugis",
     "price_per_night_inr": 9800, "rating": 4.3, "style": "mid"},
    {"name": "Hotel G Singapore", "city": "Singapore", "area": "Bencoolen",
     "price_per_night_inr": 8600, "rating": 4.2, "style": "mid"},
    {"name": "Marina Bay Sands", "city": "Singapore", "area": "Marina Bay",
     "price_per_night_inr": 34000, "rating": 4.7, "style": "luxury"},
    {"name": "Hotel Mono", "city": "Singapore", "area": "Chinatown",
     "price_per_night_inr": 10500, "rating": 4.4, "style": "mid"},
    {"name": "The Sultan", "city": "Singapore", "area": "Kampong Glam",
     "price_per_night_inr": 11200, "rating": 4.5, "style": "mid"},
]

# ---------- THINGS TO DO ----------
ACTIVITIES = [
    {"name": "Gardens by the Bay", "city": "Singapore", "interest": "city views", "hours": 3, "area": "Marina Bay"},
    {"name": "Marina Bay Sands SkyPark", "city": "Singapore", "interest": "city views", "hours": 2, "area": "Marina Bay"},
    {"name": "Singapore Flyer", "city": "Singapore", "interest": "city views", "hours": 2, "area": "Marina Bay"},
    {"name": "Chinatown Heritage Centre", "city": "Singapore", "interest": "culture", "hours": 2, "area": "Chinatown"},
    {"name": "Sultan Mosque walk", "city": "Singapore", "interest": "culture", "hours": 2, "area": "Kampong Glam"},
    {"name": "Little India walking tour", "city": "Singapore", "interest": "culture", "hours": 3, "area": "Little India"},
    {"name": "Hawker food trail", "city": "Singapore", "interest": "food", "hours": 3, "area": "Chinatown"},
    {"name": "Sentosa beach day", "city": "Singapore", "interest": "relax", "hours": 5, "area": "Sentosa"},
]

# ---------- FOOD ----------
RESTAURANTS = [
    {"name": "Maxwell Food Centre", "city": "Singapore", "cuisine": "local", "cost_inr": 500, "area": "Chinatown"},
    {"name": "Lau Pa Sat", "city": "Singapore", "cuisine": "local", "cost_inr": 600, "area": "Downtown"},
    {"name": "Komala Vilas", "city": "Singapore", "cuisine": "indian veg", "cost_inr": 700, "area": "Little India"},
    {"name": "Zam Zam", "city": "Singapore", "cuisine": "halal", "cost_inr": 800, "area": "Kampong Glam"},
    {"name": "Newton Food Centre", "city": "Singapore", "cuisine": "seafood", "cost_inr": 1600, "area": "Newton"},
    {"name": "Tiong Bahru Market", "city": "Singapore", "cuisine": "local", "cost_inr": 450, "area": "Tiong Bahru"},
]

# ---------- WEATHER ----------
WEATHER = {
    "singapore": "Hot and humid all year, 26-32 C. Short heavy rain most afternoons. Carry an umbrella.",
    "bangkok":   "Hot, 28-35 C. Heavy rain June to October.",
    "dubai":     "Very hot April to September, 35-45 C. Pleasant November to March.",
}

# ---------- LOCAL TIPS ----------
TIPS = {
    "singapore": "Indian passport holders need a visa. MRT covers the whole city. Buy an EZ-Link card. Tap water is safe. No chewing gum.",
    "bangkok":   "Visa on arrival for Indians. Use BTS Skytrain. Always agree the taxi fare first.",
    "dubai":     "Visa needed in advance. Metro is clean and cheap. Dress modestly in public places.",
}

# ---------- EXCHANGE RATES (1 unit = ? INR) ----------
RATES = {"SGD": 65.0, "USD": 88.0, "THB": 2.5, "AED": 24.0, "INR": 1.0}

print("Data loaded:", len(FLIGHTS), "flights,", len(HOTELS), "hotels,",
      len(ACTIVITIES), "activities,", len(RESTAURANTS), "restaurants.")

## 3. CrewAI tools
`crewai.tools.tool` turns typed Python functions into CrewAI tools. The name, description and argument schema guide model calls. Function bodies match the companion. The unused `retriver` alias remains for fidelity, but only seven distinct tools enter the registry.

In [ ]:



@tool
def search_flights(origin: str, destination: str, avoid_red_eye: bool = False,
                   max_price_inr: int = 999999) -> list:
    """Find flights between two cities. Set avoid_red_eye to True to remove overnight flights."""
    # FLIGHTS = await("https://jsonplaceholder.typicode.com/flights")
      # FLIGHTS = select * from flight where time = 12-2-2026
    found = [f for f in FLIGHTS
             if f["from"].lower() == origin.lower()
             and f["to"].lower() == destination.lower()
             and f["price_inr"] <= max_price_inr]
    if avoid_red_eye:
        found = [f for f in found if not f["red_eye"]]
    return sorted(found, key=lambda f: f["price_inr"])


@tool
def search_hotels(city: str, max_price_per_night_inr: int = 999999, style: str = "any") -> list:
    """Find hotels in a city. Style can be budget, mid, luxury or any."""
    found = [h for h in HOTELS
             if h["city"].lower() == city.lower()
             and h["price_per_night_inr"] <= max_price_per_night_inr]
    if style != "any":
        found = [h for h in found if h["style"] == style.lower()]
    return sorted(found, key=lambda h: -h["rating"])


@tool
def search_activities(city: str, interest: str = "any") -> list:
    """Find things to do in a city. Interest can be food, culture, city views, relax or any."""
    found = [a for a in ACTIVITIES if a["city"].lower() == city.lower()]
    if interest != "any":
        found = [a for a in found if interest.lower() in a["interest"]]
    return found


@tool
def search_restaurants(city: str, cuisine: str = "any") -> list:
    """Find places to eat in a city. Cuisine can be local, indian veg, halal, seafood or any."""
    found = [r for r in RESTAURANTS if r["city"].lower() == city.lower()]
    if cuisine != "any":
        found = [r for r in found if cuisine.lower() in r["cuisine"]]
    return sorted(found, key=lambda r: r["cost_inr"])


@tool
def retriver(city: str, cuisine: str = "any") -> list:
    """he will find the right content from document we have"""
    found = [r for r in RESTAURANTS if r["city"].lower() == city.lower()]
    if cuisine != "any":
        found = [r for r in found if cuisine.lower() in r["cuisine"]]
    return sorted(found, key=lambda r: r["cost_inr"])



@tool
def get_weather(city: str) -> str:
    """Get the usual weather for a city, so the plan can allow for rain or heat."""
    return WEATHER.get(city.lower(), "No weather information for this city.")


@tool
def get_travel_tips(city: str) -> str:
    """Get visa, transport and local rules for a city."""
    return TIPS.get(city.lower(), "No tips available for this city.")


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between currencies, for example SGD to INR."""
    if from_currency.upper() not in RATES or to_currency.upper() not in RATES:
        return "Currency not supported."
    in_inr = amount * RATES[from_currency.upper()]
    result = in_inr / RATES[to_currency.upper()]
    return f"{amount} {from_currency.upper()} = {round(result, 2)} {to_currency.upper()}"


print("8 tools defined; 7 distinct tools registered below.")

In [ ]:
search_flights.run(origin="Mumbai", destination="Singapore", avoid_red_eye=True)

In [ ]:
TOOL_REGISTRY = {
    "flight_agent":   [search_flights, convert_currency],
    "hotel_agent":    [search_hotels, convert_currency, get_travel_tips],
    "activity_agent": [search_activities, search_restaurants, get_weather],
    "critic_agent":   [],   # the critic only reads the plan, it does not search
}

# Show the registry as a small table
for agent_name, tools in TOOL_REGISTRY.items():
    tool_names = [t.name for t in tools] or ["(no tools)"]
    print(f"{agent_name:16} -> {', '.join(tool_names)}")

## 4. Agents, Tasks and Crews
An **Agent** defines a role, goal, backstory, tools and model. A **Task** supplies a concrete instruction and expected output. A **Crew** executes its Tasks and returns a `CrewOutput`; `.raw` gives text.

We use small, one-task Crews inside a Flow. Each call creates a fresh Agent, Task and Crew, avoiding retained task conversation between stages and avoiding shared mutable Agent instances across concurrent workers. Crew memory and delegation stay off. The manager is an ordinary role in this controlled workflow, not CrewAI's hierarchical manager.

`Process.hierarchical` lets a manager delegate and validate tasks autonomously. That changes who runs and how often, so it does not guarantee our exact selected-worker and single-revision policy. A Flow expresses that policy explicitly. Role/goal/backstory add CrewAI framing around the retained instructions, so generated wording can differ.


In [ ]:
AGENT_INSTRUCTIONS = {'manager_agent': 'You are the manager of a travel planning team. You never search for travel data yourself. You read the request, decide who should work, and write the final plan. Be short and clear. Never invent a flight, hotel or place that a worker did not give you.', 'flight_agent': 'You are the flight specialist. Use your tools to find real options. Never suggest a flight that did not come from a tool. Answer in exactly this shape:\nTOP 3 OPTIONS: a numbered list with airline, flight id, times and price\nRECOMMENDED: one option and one line saying why\nTRADE-OFFS: what the traveller gives up by taking it\nASSUMPTIONS: anything you had to guess', 'hotel_agent': 'You are the hotel specialist. Use your tools to find real options. Never suggest a hotel that did not come from a tool. Always check the total cost for the whole stay, not only the price per night. Answer in exactly this shape:\nTOP 3 OPTIONS: a numbered list with name, area, rating, price per night, total for the stay\nRECOMMENDED: one option and one line saying why\nTRADE-OFFS: what the traveller gives up by taking it\nASSUMPTIONS: anything you had to guess', 'activity_agent': 'You are the activities specialist. Use your tools to find real places. Never suggest a place that did not come from a tool. Do not overload a day. Two or three activities plus one meal is enough. Answer in exactly this shape:\nDAY BY DAY: for each day, morning, afternoon, evening and where to eat\nWEATHER NOTE: one line\nASSUMPTIONS: anything you had to guess', 'critic_agent': "You are a strict reviewer of travel plans. You do not rewrite the plan. You list problems only. Check four things: budget respected, days not overloaded, the traveller's stated interests actually appear, and guesses are stated honestly. Answer as a numbered list of specific fixes. Never write vague comments like 'make it better'. If the plan is fine, say NO ISSUES FOUND."}

AGENT_ROLES = {'manager_agent': 'Travel team manager', 'flight_agent': 'Flight specialist', 'hotel_agent': 'Hotel specialist', 'activity_agent': 'Activity specialist', 'critic_agent': 'Travel plan critic'}

async def ask(agent_name: str, question: str) -> str:
    """Run one fresh CrewAI task and return its raw text."""
    agent = Agent(
        role=AGENT_ROLES[agent_name],
        goal=AGENT_INSTRUCTIONS[agent_name],
        backstory="You are part of the classroom travel planning team. Follow your task contract.",
        tools=TOOL_REGISTRY.get(agent_name, []),
        llm=LLM(model="openai/" + MODEL),
        allow_delegation=False, verbose=False, max_iter=12,
    )
    task = Task(description=question,
                expected_output=AGENT_INSTRUCTIONS[agent_name], agent=agent)
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential,
                memory=False, cache=False, verbose=False, tracing=False)
    return (await crew.kickoff_async()).raw


## 5. The seven business stages
These retain the companion prompts, selection fallback and report headings. Only the ask adapter changes. Selection uses name matching and falls back to all three when no valid name appears. This preserves the original behavior rather than adding structured routing.

In [ ]:
async def manager_node(state):
    """Read the request. Write down the requirements. Decide who should work."""
    requirements = await ask('manager_agent', f"\nRead this travel request and list what you understood.\n\nREQUEST: {state['request']}\n\nList these lines only:\nFROM, TO, DAYS, TRAVELLERS, BUDGET, INTERESTS, HARD RULES, MISSING INFO\n")
    choice = await ask('manager_agent', f'\nHere are the requirements:\n{requirements}\n\nWhich specialists are needed? Choose from: flight_agent, hotel_agent, activity_agent.\nReply with the names separated by commas and nothing else.\n')
    valid = ['flight_agent', 'hotel_agent', 'activity_agent']
    workers = [name for name in valid if name in choice.lower()]
    if not workers:
        workers = valid
    print('MANAGER understood the request.')
    print('MANAGER picked:', ', '.join(workers))
    return {'requirements': requirements, 'chosen_workers': workers}

async def flight_node(state):
    """The flight agent works on its part."""
    report = await ask('flight_agent', f"Find flights for this trip.\n\n{state['requirements']}")
    return {'flight_report': report}

async def hotel_node(state):
    """The hotel agent works on its part."""
    report = await ask('hotel_agent', f"Find hotels for this trip.\n\n{state['requirements']}")
    return {'hotel_report': report}

async def activity_node(state):
    """The activity agent works on its part."""
    report = await ask('activity_agent', f"Plan the days for this trip.\n\n{state['requirements']}")
    return {'activity_report': report}

async def synthesis_node(state):
    """The manager joins the three reports into one travel plan."""
    draft = await ask('manager_agent', f"\nWrite the first version of the travel plan using ONLY the reports below.\n\nREQUIREMENTS:\n{state['requirements']}\n\nFLIGHT REPORT:\n{state.get('flight_report', 'not requested')}\n\nHOTEL REPORT:\n{state.get('hotel_report', 'not requested')}\n\nACTIVITY REPORT:\n{state.get('activity_report', 'not requested')}\n\nUse these headings:\nCHOSEN FLIGHT, CHOSEN HOTEL, DAY BY DAY PLAN, TOTAL COST, WHY THESE CHOICES, ASSUMPTIONS\n")
    print('MANAGER wrote the first plan.')
    return {'draft_plan': draft}

async def critic_node(state):
    """The critic looks for problems. It does not fix them."""
    issues = await ask('critic_agent', f"\nReview this plan against the requirements.\n\nREQUIREMENTS:\n{state['requirements']}\n\nPLAN:\n{state['draft_plan']}\n")
    print('CRITIC finished the review.')
    return {'critique': issues}

async def revision_node(state):
    """The manager fixes only what the critic listed. Everything else stays as it was."""
    if 'NO ISSUES FOUND' in state['critique'].upper():
        print('No changes needed.')
        return {'final_plan': state['draft_plan']}
    fixed = await ask('manager_agent', f"\nFix ONLY the problems listed below. Do not rewrite the parts that are fine.\n\nPLAN:\n{state['draft_plan']}\n\nPROBLEMS TO FIX:\n{state['critique']}\n\nReturn the full corrected plan, then a short section called WHAT CHANGED.\n")
    print('MANAGER fixed the plan.')
    return {'final_plan': fixed}
print('7 nodes ready.')

## 6. Native Flow state and scheduling
`Flow.state` owns the business fields. Unstructured state starts with a framework `id` plus the supplied `request`; derived fields appear when stages write them. This is a mutable dictionary, without LangGraph partial-update reducers or its conflicting-write checks.

The selected-workers listener takes a local snapshot and uses `asyncio.gather` to run separate one-task Crews concurrently. Their returned dictionaries merge only after all selected workers complete. The Flow has one worker-stage listener, not three separate graph vertices. Concurrency comes from Python orchestration around `kickoff_async`, not `Process.sequential` itself. No output-order guarantee is implied.

Flow state is business data, not a chat transcript. No `@persist` or checkpoint store is configured. CrewAI 1.15.27 creates a Flow memory facility by default, but this pipeline never calls remember or recall and never passes it to its Crews. A fresh Flow isolates every trip. A failure stops the run and propagates to the caller.


In [ ]:
class TravelPlannerFlow(Flow):
    @start()
    async def manager(self):
        self.state.update(await manager_node(self.state))

    @listen(manager)
    async def selected_workers(self):
        nodes = {"flight_agent": flight_node, "hotel_agent": hotel_node,
                 "activity_agent": activity_node}
        snapshot = dict(self.state)
        updates = await asyncio.gather(
            *(nodes[name](snapshot) for name in snapshot["chosen_workers"]))
        for update in updates:
            self.state.update(update)

    @listen(selected_workers)
    async def synthesis(self):
        self.state.update(await synthesis_node(self.state))

    @listen(synthesis)
    async def critic(self):
        self.state.update(await critic_node(self.state))

    @listen(critic)
    async def revision(self):
        self.state.update(await revision_node(self.state))
        return dict(self.state)

async def run_travel_state(request: str) -> dict:
    return await TravelPlannerFlow().kickoff_async(inputs={"request": request})


## 7. Run the same demo
Use top-level `await`. `plan_trip` returns the full state dictionary, correcting the original LangGraph wrapper’s `fina` typo. The final text is `result["final_plan"]`. A no-issues review skips the revision model call. Otherwise exactly one revision runs, without another critic pass. The inherited sentinel test is substring-based and can misread a negated phrase.

In [ ]:
async def plan_trip(request: str, show_details: bool = True):
    """Run the full agent team on a travel request."""
    print("=" * 70)
    print("REQUEST:", request)
    print("=" * 70)

    result = await run_travel_state(request)

    if show_details:
        print("\n" + "-" * 70)
        print("WHAT THE MANAGER UNDERSTOOD")
        print("-" * 70)
        print(result["requirements"])

        for key, title in [("flight_report", "FLIGHT AGENT"),
                           ("hotel_report", "HOTEL AGENT"),
                           ("activity_report", "ACTIVITY AGENT")]:
            if result.get(key):
                print("\n" + "-" * 70)
                print(title)
                print("-" * 70)
                print(result[key])

        print("\n" + "-" * 70)
        print("FIRST DRAFT")
        print("-" * 70)
        print(result["draft_plan"])

        print("\n" + "-" * 70)
        print("WHAT THE CRITIC FOUND")
        print("-" * 70)
        print(result["critique"])

    print("\n" + "=" * 70)
    print("FINAL TRAVEL PLAN")
    print("=" * 70)
    print(result["final_plan"])
    return result


print("Ready to run.")

In [ ]:
demo_request = """Plan a 4-day Singapore trip from Mumbai for two people.
Mid-budget. Avoid red-eye flights.
Prefer food, city views and cultural sites.
Keep hotel budget under 45000 rupees."""

result = await plan_trip(demo_request)

## 8. Inspect grounding
The inherited checks below display known names present in the plan. They cannot detect all invented names or validate arithmetic. Inspect traveller counts, nights, arrival-day timing, excluded costs and assumptions too.

In [ ]:
# Check that every flight id in the final plan really exists in our data
plan_text = result["final_plan"]

real_ids = [f["id"] for f in FLIGHTS]
found_ids = [fid for fid in real_ids if fid in plan_text]
print("Real flight ids used in the plan:", found_ids or "none")

real_hotels = [h["name"] for h in HOTELS]
found_hotels = [name for name in real_hotels if name in plan_text]
print("Real hotel names used in the plan:", found_hotels or "none")

print("\nIf both lines are empty, the plan was invented and the grounding check has failed.")

## 9. Your trip
A new request creates a fresh Flow and fresh Crews. Unsupported routes can return empty tool results. Agents should explain missing inventory instead of inventing options.

In [ ]:
your_request = input("Describe your trip: ")
result = await plan_trip(your_request)

## Framework comparison and exercises

| Concern | LangGraph source | This CrewAI version |
|---|---|---|
| Agent loop | LangChain create_agent | CrewAI Agent executing a Task in a Crew |
| Scheduling | StateGraph edges | Flow start/listen methods |
| Workers | Parallel graph nodes | asyncio.gather around independent Crews in one listener |
| State | TypedDict and partial updates | Native mutable Flow.state dictionary plus UUID |
| Reports | Separate report keys | Same keys, merged at an explicit barrier |
| Result | invoke result dictionary | kickoff_async result from the last Flow method |
| Revision | Conditional model-call skip | Same skip inside final listener |

**Exercises:** request hotels only and inspect absent report fields. Ask for an unsupported city. Change a tool filter, then compare its direct output and specialist report. Explain why a CrewOutput is different from Flow state. Explain why a saved final plan does not provide crash recovery.

## Official references
Verified against CrewAI 1.15.27 (Python >=3.10,<3.14).
- [Flows and state](https://docs.crewai.com/en/concepts/flows)
- [Agents](https://docs.crewai.com/en/concepts/agents)
- [Tasks and task outputs](https://docs.crewai.com/en/concepts/tasks)
- [Crews and kickoff](https://docs.crewai.com/en/concepts/crews)
- [Tools](https://docs.crewai.com/en/concepts/tools)
- [Hierarchical process](https://docs.crewai.com/en/learn/hierarchical-process)
- [Package metadata](https://pypi.org/project/crewai/1.15.27/)

Developer validation is separate in `../scripts/validate_crewai_travel_planner.py`. It uses a deterministic CrewAI BaseLLM to exercise actual Agents, Tasks, Crews, tools and Flow routing without paid calls. It does not establish real OpenAI authentication, hosted Colab installation, model-generated quality or real-world travel accuracy. All saved code outputs are cleared.
